# Logistic regression & binary cross-entropy

Run cells in order. All examples use Python’s standard library and tiny synthetic data. The outputs are teaching examples, not held-out performance estimates.

In [1]:
import math

def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    exp_z = math.exp(z)
    return exp_z / (1 + exp_z)

def bce(labels, probabilities):
    return -sum(y*math.log(p) + (1-y)*math.log1p(-p)
                for y, p in zip(labels, probabilities)) / len(labels)

print('sigmoid(-2), sigmoid(0), sigmoid(2):',
      [round(sigmoid(z), 4) for z in (-2, 0, 2)])

sigmoid(-2), sigmoid(0), sigmoid(2): [0.1192, 0.5, 0.8808]


## Derive one gradient step

For $x=[0,1]$, $y=[0,1]$, and zero initial parameters, the probabilities are both 0.5. The gradient of mean BCE with respect to the weight is $\sum_i(p_i-y_i)x_i/n$.

In [2]:
xs, ys = [0.0, 1.0], [0, 1]
w, b, learning_rate = 0.0, 0.0, 1.0
probs = [sigmoid(w*x+b) for x in xs]
grad_w = sum((p-y)*x for x, y, p in zip(xs, ys, probs)) / len(xs)
grad_b = sum(p-y for y, p in zip(ys, probs)) / len(xs)
print('initial BCE:', round(bce(ys, probs), 4))
print('grad_w, grad_b:', round(grad_w, 4), round(grad_b, 4))
w -= learning_rate * grad_w
b -= learning_rate * grad_b
updated = [sigmoid(w*x+b) for x in xs]
print('new weight, probabilities:', round(w, 4), [round(p, 4) for p in updated])
print('new BCE:', round(bce(ys, updated), 4))

initial BCE: 0.6931
grad_w, grad_b: -0.25 0.0
new weight, probabilities: 0.25 [0.5, 0.5622]
new BCE: 0.6345


## Check the derivative and decision threshold

A finite difference checks the sign and scale of an analytic gradient. The threshold affects labels, not the fitted probabilities.

In [3]:
epsilon = 1e-5
def loss_at(weight):
    return bce(ys, [sigmoid(weight*x) for x in xs])
finite_difference = (loss_at(epsilon) - loss_at(-epsilon)) / (2*epsilon)
print('analytic vs numeric:', round(grad_w, 6), round(finite_difference, 6))
for threshold in (0.5, 0.6):
    predicted = [int(p >= threshold) for p in updated]
    tp = sum(y == 1 and pred == 1 for y, pred in zip(ys, predicted))
    fp = sum(y == 0 and pred == 1 for y, pred in zip(ys, predicted))
    fn = sum(y == 1 and pred == 0 for y, pred in zip(ys, predicted))
    print(f'threshold {threshold}: predicted={predicted}, TP={tp}, FP={fp}, FN={fn}')

analytic vs numeric: -0.25 -0.25
threshold 0.5: predicted=[1, 1], TP=1, FP=1, FN=0
threshold 0.6: predicted=[0, 0], TP=0, FP=0, FN=1


### Try it

Change the learning rate or a label and recompute the gradient. On real data, separate training, validation, and test sets before tuning regularization or a threshold. Check calibration as well as classification metrics.